# Sentimiento de reseñas de películas: Evaluación

Clasificar reseñas en inglés como positivas o negativas.

F1 macro, accuracy y métricas por clase. El test deduplicado puede tener menos filas que el oficial; los conteos quedan en preparación.

Consulta la [metodología](../docs/metodologia.md). Los reportes corresponden a la última ejecución de cada etapa.


In [1]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("imdb_sentiment")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from imdb_sentiment.alerts import configurar_notebook, mostrar_alertas
configurar_notebook()


In [2]:
from imdb_sentiment.flow import run_stage
metricas, comentario = run_stage(PROJECT_ROOT, "evaluacion")
display(Markdown(comentario))
from imdb_sentiment.presentation import display_evaluation
display_evaluation(PROJECT_ROOT, metricas)


# Sentimiento de reseñas de películas: evaluacion

En **24,799 textos de test**, el modelo obtiene **F1 macro 0.8800** y supera el baseline (0.3340). Consulta la matriz de confusión y las métricas por clase; el desempeño mide sentimiento de reseñas en inglés, no gusto personal por películas.


,baseline,modelo
casos,24799.000000,24799.000000
accuracy,0.501593,0.879995
f1_macro,0.334040,0.879994
etiquetas_fuera_vocabulario_train,0.000000,0.000000


,precision,recall,f1-score,support
negativo,0.879735,0.879450,0.879592,12360.000000
positivo,0.880254,0.880537,0.880395,12439.000000
accuracy,0.879995,0.879995,0.879995,0.879995
macro avg,0.879994,0.879993,0.879994,24799.000000
weighted avg,0.879995,0.879995,0.879995,24799.000000


## Alertas

Avisos agrupados con descripción y acción sugerida. Los errores no se ocultan; no registrar warnings no garantiza que el análisis sea válido.

In [ ]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/05_evaluacion.json")
from imdb_sentiment.informe import generar_informe
generar_informe(PROJECT_ROOT)
